In [1]:
# =====================================================================
# Stage 1: imports, constants and configuration
# =====================================================================
import json
import math
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import requests                         # only used by the optional live-data cell
from matplotlib.colors import to_hex

# Physical constants
MU_EARTH_KM3_S2 = 398600.4418           # Earth gravitational parameter [km^3/s^2]
EARTH_RADIUS_KM = 6378.137              # Earth equatorial radius [km], used as a sphere

# Workspace folders
BASE_DIR = Path('/content/satellite_orbits')
IN_DIR = BASE_DIR / 'IN'
OUT_DIR = BASE_DIR / 'OUT'
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
STEP_SECONDS = 5                        # propagation time step [s]
DURATION_HOURS = 24                     # propagation window [h]
LIVE_GROUP = None                       # optional extension: e.g. 'GALILEO' for live CelesTrak data

print(f"Window: {DURATION_HOURS} h at {STEP_SECONDS} s steps | live data: {LIVE_GROUP or 'off (workshop snapshot)'}")
print(f"IN_DIR:  {IN_DIR}\nOUT_DIR: {OUT_DIR}")

ModuleNotFoundError: No module named 'matplotlib'

### Load GP JSON    


In [ ]:
# =====================================================================
# Stage 2: get the input files and load the GP records
# =====================================================================
INPUT_FILES = ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']
UPLOAD_DIRS = [Path('/content')]              # where the Colab Files panel puts dragged files
GP_FIELDS = ['OBJECT_NAME', 'NORAD_CAT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
             'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY']


def find_input_zip():
    """Return the first complete orbit_workshop_inputs*.zip in UPLOAD_DIRS, or None."""
    for folder in UPLOAD_DIRS:
        for path in sorted(folder.glob('orbit_workshop_inputs*.zip')):
            if zipfile.is_zipfile(path):        # False while the file is still uploading
                return path
    return None


def extract_inputs(timeout_s=900):
    """Wait for the input ZIP, then copy its three known files into IN_DIR (skipped if already there)."""
    if all((IN_DIR / name).exists() for name in INPUT_FILES):
        return
    print("Drag orbit_workshop_inputs.zip into the /content folder of the Files panel (folder icon, left).")
    print("This cell continues by itself as soon as the ZIP has arrived...")
    zip_path, waited = find_input_zip(), 0
    while zip_path is None:
        if waited >= timeout_s:
            raise FileNotFoundError("orbit_workshop_inputs.zip not found after 15 minutes: add it and run this cell again.")
        time.sleep(2)
        waited += 2
        zip_path = find_input_zip()
    with zipfile.ZipFile(zip_path) as zf:
        for name in INPUT_FILES:                # extract only the files we know
            (IN_DIR / name).write_bytes(zf.read(name))
    print(f"Extracted {len(INPUT_FILES)} files from {zip_path.name}.")


def read_gp_json(path):
    """Load a GP JSON file (a list of OMM records) and check that every record has the fields we use."""
    records = json.loads(Path(path).read_text(encoding='utf-8'))
    if not isinstance(records, list) or not records:
        raise ValueError(f"{path} must contain a non-empty JSON list of GP records.")
    for rec in records:
        missing = [k for k in GP_FIELDS if k not in rec]
        if missing:
            raise ValueError(f"{rec.get('OBJECT_NAME', '?')} is missing the fields {missing}.")
    return records


def print_sats(sats):
    for s in sats:
        print(f"  {s['OBJECT_NAME']:<24} NORAD {s['NORAD_CAT_ID']:<7} EPOCH {s['EPOCH']}")


extract_inputs()
sats = read_gp_json(IN_DIR / 'orbital_elements.json')
data_manifest = json.loads((IN_DIR / 'data_manifest.json').read_text(encoding='utf-8'))
source_note = f"Workshop snapshot: native CelesTrak GP JSON retrieved {data_manifest['retrieved_utc']}"
print(source_note)
print_sats(sats)

### Parse Elements

In [ ]:
# =====================================================================
# Stage 3: GP record -> Kepler elements in model units
# =====================================================================
def parse_gp_epoch(text):
    """CelesTrak epochs are ISO-8601 UTC without an offset, e.g. '2026-11-10T04:10:34.900320'."""
    return datetime.fromisoformat(text.rstrip('Z')).replace(tzinfo=timezone.utc)


def parse_gp_elements(rec):
    """Convert one GP record into the elements used by the model (km, rad, rad/s, UTC)."""
    n = 2 * math.pi * rec['MEAN_MOTION'] / 86400              # mean motion: rev/day -> rad/s
    name, norad_id = rec['OBJECT_NAME'], int(rec['NORAD_CAT_ID'])
    return {
        'name': name,
        'norad_id': norad_id,
        'label': f"{name} [{norad_id}]",
        'epoch': parse_gp_epoch(rec['EPOCH']),
        'n_rad_s': n,
        'a_km': (MU_EARTH_KM3_S2 / n ** 2) ** (1 / 3),        # Kepler's third law
        'e': rec['ECCENTRICITY'],                              # already a decimal number
        'i_rad': math.radians(rec['INCLINATION']),
        'raan_rad': math.radians(rec['RA_OF_ASC_NODE']),
        'argp_rad': math.radians(rec['ARG_OF_PERICENTER']),
        'M0_rad': math.radians(rec['MEAN_ANOMALY']),
    }


def elements_table(elements):
    """Text table of the elements (printed here, reused in the Stage 8 report)."""
    lines = [f"{'Satellite':<24} {'NORAD':>6}  {'Epoch (UTC)':<19} {'a [km]':>9} {'e':>9} "
             f"{'i':>7} {'RAAN':>7} {'argp':>7} {'M0':>7}  (angles in deg)"]
    for el in elements:
        angles = [math.degrees(el[k]) for k in ('i_rad', 'raan_rad', 'argp_rad', 'M0_rad')]
        lines.append(f"{el['name']:<24} {el['norad_id']:>6}  {el['epoch']:%Y-%m-%d %H:%M:%S} {el['a_km']:>9.1f} "
                     f"{el['e']:>9.6f} " + ' '.join(f"{x:>7.2f}" for x in angles))
    return lines


elements = [parse_gp_elements(s) for s in sats]
print('\n'.join(elements_table(elements)))

### Time Window

In [ ]:
# =====================================================================
# Stage 4: one common UTC time grid
# =====================================================================
newest_epoch = max(el['epoch'] for el in elements)
start = newest_epoch.replace(hour=0, minute=0, second=0, microsecond=0)    # UTC midnight of the newest record
end = start + timedelta(hours=DURATION_HOURS)
n_points = int((end - start).total_seconds()) // STEP_SECONDS + 1          # +1: include both ends
times = [start + timedelta(seconds=k * STEP_SECONDS) for k in range(n_points)]

print(f"Window (UTC): {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} | step {STEP_SECONDS} s | "
      f"{n_points} points per satellite\n")
print(f"{'Satellite':<24} {'Offset start - epoch [days]':>28}")
for el in elements:
    offset_days = (start - el['epoch']).total_seconds() / 86400
    flag = '   <-- more than 7 days: stale record' if abs(offset_days) > 7 else ''
    print(f"{el['name']:<24} {offset_days:>28.2f}{flag}")

### Propagate the orbits

In [ ]:
# =====================================================================
# Stage 5: the physics - Kepler's equation, Earth rotation, propagation
# =====================================================================
J2000 = datetime(2000, 1, 1, 12, tzinfo=timezone.utc)


def solve_kepler(M, e, tol=1e-12):
    """Solve Kepler's equation M = E - e sin E for E with Newton-Raphson."""
    E = M if e < 0.8 else math.pi                       # good starting guess
    for _ in range(50):
        step = (E - e * math.sin(E) - M) / (1 - e * math.cos(E))
        E -= step
        if abs(step) < tol:
            return E
    raise RuntimeError(f"Kepler's equation did not converge (M={M}, e={e}).")


def gmst_rad(t):
    """Greenwich Mean Sidereal Time [rad] = Earth's rotation angle (IAU-1982, Meeus eq. 12.4)."""
    d = (t - J2000).total_seconds() / 86400             # days since J2000.0
    T = d / 36525                                       # Julian centuries
    theta_deg = 280.46061837 + 360.98564736629 * d + 0.000387933 * T ** 2 - T ** 3 / 38710000
    return math.radians(theta_deg % 360)


def propagate(el, times):
    """Two-body propagation of one satellite: one point (time, lat, lon, alt, speed) per UTC time."""
    a, e, i, raan = el['a_km'], el['e'], el['i_rad'], el['raan_rad']
    points = []
    for t in times:
        # 1. mean anomaly at t
        M = (el['M0_rad'] + el['n_rad_s'] * (t - el['epoch']).total_seconds()) % (2 * math.pi)
        # 2. Kepler's equation -> eccentric anomaly
        E = solve_kepler(M, e)
        # 3. radius, true anomaly and argument of latitude
        r = a * (1 - e * math.cos(E))
        nu = math.atan2(math.sqrt(1 - e ** 2) * math.sin(E), math.cos(E) - e)
        u = el['argp_rad'] + nu
        # 4. inertial (ECI-like) position [km]
        x = r * (math.cos(raan) * math.cos(u) - math.sin(raan) * math.sin(u) * math.cos(i))
        y = r * (math.sin(raan) * math.cos(u) + math.cos(raan) * math.sin(u) * math.cos(i))
        z = r * math.sin(u) * math.sin(i)
        # 5. rotate by the Earth's angle -> Earth-fixed (ECEF) axes
        theta = gmst_rad(t)
        x_ecef = math.cos(theta) * x + math.sin(theta) * y
        y_ecef = -math.sin(theta) * x + math.cos(theta) * y
        # 6. spherical latitude, longitude and altitude
        lat = math.degrees(math.atan2(z, math.hypot(x_ecef, y_ecef)))
        lon = math.degrees(math.atan2(y_ecef, x_ecef))
        # 7. speed from the vis-viva equation
        speed = math.sqrt(MU_EARTH_KM3_S2 * (2 / r - 1 / a))
        points.append({'time': t, 'lat_deg': lat, 'lon_deg': lon, 'alt_km': r - EARTH_RADIUS_KM, 'speed_km_s': speed})
    return points


# Checks on the two helpers
assert abs(math.degrees(gmst_rad(J2000)) - 280.46061837) < 1e-6           # Earth's angle at J2000.0 is not zero
for e_test in (0.0, 0.07, 0.7):
    for M_test in (0.1, 1.0, 3.0, 6.0):
        E_test = solve_kepler(M_test, e_test)
        assert abs(E_test - e_test * math.sin(E_test) - M_test) < 1e-10   # Kepler's equation is satisfied
print(f"Helpers OK: GMST(J2000) = {math.degrees(gmst_rad(J2000)):.5f} deg, Kepler residuals < 1e-10.")


# =====================================================================
# Stage 5: propagate every satellite over the common grid
# =====================================================================
tracks = [propagate(el, times) for el in elements]         # same order as elements

for el, pts in zip(elements, tracks):
    print(el['label'])
    for p in (pts[0], pts[len(pts) // 2], pts[-1]):         # first, middle and last point
        print(f"  {p['time']:%Y-%m-%d %H:%M} UTC | lat {p['lat_deg']:7.2f} | lon {p['lon_deg']:8.2f} deg | "
              f"alt {p['alt_km']:8.1f} km | speed {p['speed_km_s']:.3f} km/s")

# Physical check: SES-5 is geostationary, parked near 5 deg East above the equator
for el, pts in zip(elements, tracks):
    if el['norad_id'] == 38652:
        lons = [p['lon_deg'] for p in pts]
        print(f"\nSES-5 longitude over the window: {min(lons):.2f} to {max(lons):.2f} deg (expected near +5 deg E)")
        assert 4 < min(lons) and max(lons) < 6, "SES-5 is not near 5 deg E: check the GMST formula and the rotation signs"

### Orbit Summary

In [ ]:
# =====================================================================
# Stage 6: orbit summary
# =====================================================================
def orbit_summary(el, pts):
    speeds = [p['speed_km_s'] for p in pts]
    return {
        'name': el['name'], 'norad_id': el['norad_id'],
        'period_h': 2 * math.pi / el['n_rad_s'] / 3600,
        'perigee_km': el['a_km'] * (1 - el['e']) - EARTH_RADIUS_KM,
        'apogee_km': el['a_km'] * (1 + el['e']) - EARTH_RADIUS_KM,
        'e': el['e'], 'i_deg': math.degrees(el['i_rad']),
        'speed_min': min(speeds), 'speed_max': max(speeds),
    }


def summary_table(summaries):
    """Text table of the summary (printed here, reused in the Stage 8 report)."""
    lines = [f"{'Satellite':<24} {'NORAD':>6} {'Period [h]':>10} {'Perigee [km]':>12} {'Apogee [km]':>11} "
             f"{'e':>9} {'i [deg]':>7} {'Speed min - max [km/s]':>22}"]
    for s in summaries:
        lines.append(f"{s['name']:<24} {s['norad_id']:>6} {s['period_h']:>10.3f} {s['perigee_km']:>12.1f} "
                     f"{s['apogee_km']:>11.1f} {s['e']:>9.6f} {s['i_deg']:>7.2f} "
                     f"{s['speed_min']:>14.3f} - {s['speed_max']:.3f}")
    return lines


summaries = [orbit_summary(el, pts) for el, pts in zip(elements, tracks)]
print('\n'.join(summary_table(summaries)))

# Physical check: Galileo orbits in about 14 h 05 min
for s in summaries:
    if s['norad_id'] == 59598:
        assert 14.0 < s['period_h'] < 14.2, f"Galileo period {s['period_h']:.2f} h: check the mean-motion conversion"

### Visualization

In [ ]:
# =====================================================================
# Stage 7: ground-track and speed plots
# =====================================================================
COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']   # one colour per satellite (reused in the KML)


def load_coastline(path):
    """world_map.txt: 'latitude longitude' per line (latitude FIRST), blank lines between segments."""
    segments, current = [], []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if line.strip():
            lat, lon = map(float, line.split()[:2])
            current.append((lon, lat))                  # store as (x, y) = (lon, lat)
        elif current:
            segments.append(current)
            current = []
    if current:
        segments.append(current)
    return segments


# Figure 1: ground tracks on top of the coastline
fig, ax = plt.subplots(figsize=(12, 6))
for segment in load_coastline(IN_DIR / 'world_map.txt'):
    lons, lats = zip(*segment)
    ax.plot(lons, lats, color='gray', linewidth=0.6)
for idx, (el, pts) in enumerate(zip(elements, tracks)):
    ax.scatter([p['lon_deg'] for p in pts], [p['lat_deg'] for p in pts], s=2, color=COLORS[idx], label=el['label'])
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel='Longitude [deg]', ylabel='Latitude (geocentric) [deg]',
       title='Ground tracks (two-body model, spherical Earth)')
ax.grid(linestyle=':', alpha=0.5)
ax.legend(loc='upper right', markerscale=5, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / 'ground_tracks.png', dpi=200)
plt.show()

# Figure 2: speed vs time, one panel per satellite
fig, axes = plt.subplots(len(tracks), 1, figsize=(12, 2.6 * len(tracks) + 1), sharex=True, squeeze=False)
for idx, (el, pts) in enumerate(zip(elements, tracks)):
    ax = axes[idx, 0]
    ax.plot([p['time'] for p in pts], [p['speed_km_s'] for p in pts], color=COLORS[idx], linewidth=1.8)
    ax.set_title(el['label'], fontsize=10, loc='left')
    ax.set_ylabel('Speed [km/s]')
    ax.ticklabel_format(axis='y', useOffset=False)      # show real values, not an offset
    ax.grid(linestyle=':', alpha=0.5)
axes[-1, 0].set_xlabel('Time (UTC)')
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUT_DIR / 'velocity_timeseries.png', dpi=200)
plt.show()

### Google Earth KML

In [ ]:
# =====================================================================
# Stage 8: Google Earth files (KML)
# =====================================================================
KML_NS = 'http://www.opengis.net/kml/2.2'
GX_NS = 'http://www.google.com/kml/ext/2.2'               # Google extension used for the animation
ET.register_namespace('', KML_NS)
ET.register_namespace('gx', GX_NS)
ICON_URL = 'http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png'   # white icon, tinted per satellite
ANIMATION_STEP_SECONDS = 60                              # one animation point per minute keeps the file light


def kml(tag):
    return f'{{{KML_NS}}}{tag}'


def gx(tag):
    return f'{{{GX_NS}}}{tag}'


def kml_color(idx):
    """Figure colour -> KML colour. KML writes alpha, blue, green, red: tab:blue #1f77b4 -> ffb4771f."""
    rgb = to_hex(COLORS[idx]).lstrip('#')
    return 'ff' + rgb[4:6] + rgb[2:4] + rgb[0:2]


def new_kml_document(title, description):
    """<kml><Document> with one style per satellite, in the same colours as the figures."""
    root = ET.Element(kml('kml'))
    doc = ET.SubElement(root, kml('Document'))
    ET.SubElement(doc, kml('name')).text = title
    ET.SubElement(doc, kml('description')).text = description
    for idx in range(len(elements)):
        style = ET.SubElement(doc, kml('Style'), id=f'sat{idx}')
        icon_style = ET.SubElement(style, kml('IconStyle'))
        ET.SubElement(icon_style, kml('color')).text = kml_color(idx)
        ET.SubElement(ET.SubElement(icon_style, kml('Icon')), kml('href')).text = ICON_URL
        line_style = ET.SubElement(style, kml('LineStyle'))
        ET.SubElement(line_style, kml('color')).text = kml_color(idx)
        ET.SubElement(line_style, kml('width')).text = '2'
    return root, doc


def add_placemark(doc, idx, name):
    placemark = ET.SubElement(doc, kml('Placemark'))
    ET.SubElement(placemark, kml('name')).text = name
    ET.SubElement(placemark, kml('styleUrl')).text = f'#sat{idx}'
    return placemark


def split_at_dateline(pts):
    """Cut a track wherever the longitude jumps across +/-180 deg."""
    segments = [[pts[0]]]
    for prev, p in zip(pts, pts[1:]):
        if abs(p['lon_deg'] - prev['lon_deg']) > 180:
            segments.append([])
        segments[-1].append(p)
    return segments


# 1. Earth-fixed 3D orbits: "lon,lat,alt" with the altitude in METRES
root, doc = new_kml_document('Multi-Satellite Orbits | Earth-fixed',
                             'Two-body model, spherical Earth: approximate educational paths.')
for idx, (el, pts) in enumerate(zip(elements, tracks)):
    line = ET.SubElement(add_placemark(doc, idx, el['label']), kml('LineString'))
    ET.SubElement(line, kml('altitudeMode')).text = 'absolute'
    ET.SubElement(line, kml('coordinates')).text = ' '.join(
        f"{p['lon_deg']:.5f},{p['lat_deg']:.5f},{p['alt_km'] * 1000:.0f}" for p in pts)
ET.ElementTree(root).write(OUT_DIR / 'orbit_tracks_ecef.kml', encoding='utf-8', xml_declaration=True)

# 2. Animation: timestamped ground-track points, played with Google Earth's time slider
stride = max(1, ANIMATION_STEP_SECONDS // STEP_SECONDS)
root, doc = new_kml_document('Multi-Satellite Ground Tracks | Animation',
                             'Press play on the Google Earth time slider (best in Google Earth Pro).')
for idx, (el, pts) in enumerate(zip(elements, tracks)):
    multi_track = ET.SubElement(add_placemark(doc, idx, el['label']), gx('MultiTrack'))
    for segment in split_at_dateline(pts[::stride]):
        track = ET.SubElement(multi_track, gx('Track'))
        ET.SubElement(track, kml('altitudeMode')).text = 'clampToGround'
        for p in segment:                                # first all the timestamps...
            ET.SubElement(track, kml('when')).text = f"{p['time']:%Y-%m-%dT%H:%M:%SZ}"
        for p in segment:                                # ...then the matching positions "lon lat alt"
            ET.SubElement(track, gx('coord')).text = f"{p['lon_deg']:.5f} {p['lat_deg']:.5f} 0"
ET.ElementTree(root).write(OUT_DIR / 'orbit_animation.kml', encoding='utf-8', xml_declaration=True)
print("Written: orbit_tracks_ecef.kml, orbit_animation.kml")


# =====================================================================
# Stage 8: text report and output ZIP
# =====================================================================
report = [
    'Satellite Orbit Report - built during Space Career Week 2026, GNSS Academy', '',
    f'Data:   {source_note}',
    f'Window: {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} UTC, {STEP_SECONDS} s steps, {n_points} points per satellite',
    'Model:  two-body Kepler propagation of GP mean elements, spherical Earth, GMST rotation, no drag, no J2.',
    '        Educational approximation, not precision orbit prediction.', '',
    'ORBIT SUMMARY', *summary_table(summaries), '',
    'ORBITAL ELEMENTS AT EPOCH', *elements_table(elements),
]
(OUT_DIR / 'orbit_report.txt').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('\n'.join(report))

DELIVERABLES = ['ground_tracks.png', 'velocity_timeseries.png', 'orbit_tracks_ecef.kml', 'orbit_animation.kml',
                'orbit_report.txt']
zip_path = OUT_DIR / 'satellite_orbit_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in DELIVERABLES:
        zf.write(OUT_DIR / name, arcname=name)
print(f"\n{zip_path.name}:")
for name in DELIVERABLES:
    print(f"  {name:<26} {(OUT_DIR / name).stat().st_size / 1024:>7.0f} KB")


# Separate cell so that "Run all" does not trigger a download
from google.colab import files
files.download(str(zip_path))